## Scraping

In [ ]:
import requests
import base64
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

BASE_URL = "https://www.singaporepools.com.sg/en/product/pages/4d_results.aspx?sppl="

session = requests.Session()

rows = []

START_DRAW = 5452
STOP_DRAW = 4983

draw = START_DRAW
fail_count = 0

while draw >= STOP_DRAW:

    encoded = base64.b64encode(f"DrawNumber={draw}".encode()).decode()
    url = BASE_URL + encoded

    print(f"Scraping draw {draw}")

    try:
        r = session.get(url, timeout=10)
    except requests.RequestException:
        fail_count += 1
        if fail_count > 20:
            break
        draw -= 1
        continue

    if r.status_code != 200:
        fail_count += 1
        if fail_count > 20:
            break
        draw -= 1
        continue

    soup = BeautifulSoup(r.text, "html.parser")

    # -------------------------
    # Extract draw date
    # -------------------------

    draw_date = None
    page_text = soup.get_text(" ", strip=True)

    date_match = re.search(
        r"\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun),\s\d{2}\s\w{3}\s\d{4}",
        page_text
    )

    if date_match:
        try:
            draw_date = pd.to_datetime(date_match.group(0))
        except:
            draw_date = None

    # -------------------------
    # Extract 4D numbers
    # -------------------------

    numbers = []

    for td in soup.find_all("td"):

        txt = td.get_text(strip=True)

        if txt.isdigit() and len(txt) == 4:
            numbers.append(txt)

    # Ensure correct count
    if len(numbers) < 23:

        fail_count += 1

        if fail_count > 20:
            print("Too many failures, stopping")
            break

        draw -= 1
        continue

    fail_count = 0

    row = {
        "draw_number": draw,
        "draw_date": draw_date,

        "first_prize": numbers[0],
        "second_prize": numbers[1],
        "third_prize": numbers[2],

        "starter_1": numbers[3],
        "starter_2": numbers[4],
        "starter_3": numbers[5],
        "starter_4": numbers[6],
        "starter_5": numbers[7],
        "starter_6": numbers[8],
        "starter_7": numbers[9],
        "starter_8": numbers[10],
        "starter_9": numbers[11],
        "starter_10": numbers[12],

        "consolation_1": numbers[13],
        "consolation_2": numbers[14],
        "consolation_3": numbers[15],
        "consolation_4": numbers[16],
        "consolation_5": numbers[17],
        "consolation_6": numbers[18],
        "consolation_7": numbers[19],
        "consolation_8": numbers[20],
        "consolation_9": numbers[21],
        "consolation_10": numbers[22],
    }

    rows.append(row)

    draw -= 1

    time.sleep(0.2)

# -------------------------
# Convert to dataframe
# -------------------------

df = pd.DataFrame(rows)

# Convert draw date
df["draw_date"] = pd.to_datetime(df["draw_date"])

# Remove duplicates
df = df.drop_duplicates(subset="draw_number")

# Sort
df = df.sort_values("draw_number")

# -------------------------
# Save CSV
# -------------------------

df.to_csv("singapore_4d_history.csv", index=False)

print("Scraping complete")
print("Total draws saved:", len(df))

## Machine Learning 

In [1]:
import pandas as pd # Load data manipulation library
import matplotlib.pyplot as plt # Load data visualization library
import seaborn as sns # Load statistical data visualization library
import numpy as np # Load numerical computation library

# Initialize machine learning models
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score , root_mean_squared_error
import joblib # Used to save models

In [2]:
df = pd.read_csv('singapore_4d_history.csv')
df

,draw_number,draw_date,first_prize,second_prize,third_prize,starter_1,starter_2,starter_3,starter_4,starter_5,...,consolation_1,consolation_2,consolation_3,consolation_4,consolation_5,consolation_6,consolation_7,consolation_8,consolation_9,consolation_10
0,4983,5/3/2023,8013,8140,2276,72,332,903,1623,2893,...,455,1441,1809,2650,3486,5487,6404,7751,9512,9930
1,4984,8/3/2023,7493,8400,1971,1318,2126,3438,4030,4921,...,1671,2339,2530,2829,3666,4690,4737,5187,8288,8732
2,4985,11/3/2023,7982,4760,212,342,613,1404,4645,5956,...,1073,3586,3648,4233,5933,6269,7026,7154,8558,9256
3,4986,12/3/2023,691,2922,1822,1221,2099,3878,4006,4162,...,972,1279,3310,3321,3424,4343,5862,5952,9139,9785
4,4987,15/3/2023,1905,8878,5744,741,1428,2654,3198,3573,...,250,277,677,804,1030,1535,2762,3114,4705,8981
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,5448,22/2/2026,1903,7215,3568,41,1203,1695,3950,6171,...,1376,1790,2634,3301,4698,5217,5685,7875,9419,9984
466,5449,25/2/2026,1516,7309,2708,814,1068,3542,3847,4163,...,1511,2357,3115,4586,4965,6034,6445,6473,7266,8196
467,5450,28/2/2026,4172,198,5482,1002,3679,4256,4411,4486,...,433,515,2090,2546,4101,4473,4558,6202,8441,9542
468,5451,1/3/2026,3927,7192,2607,318,871,1083,1698,1854,...,483,1995,2792,4408,6217,7338,7340,7953,9154,9595


In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
import os
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.utils import to_categorical


# -----------------------
# CONFIG
# -----------------------

CSV_FILE = "singapore_4d_history.csv"
RESULTS_URL = "https://www.singaporepools.com.sg/en/product/Pages/4d_results.aspx"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

np.random.seed(42)

# -----------------------
# HELPER
# -----------------------

def normalize_probs(p):
    p = np.array(p)
    p = np.maximum(p, 0)
    s = p.sum()
    if s == 0:
        return np.ones(len(p)) / len(p)
    return p / s


# -----------------------
# GET LATEST WEBSITE DATE
# -----------------------

def get_latest_website_date():

    try:

        r = requests.get(RESULTS_URL, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        date_element = soup.find(
            "span",
            {"id":"ctl00_ctl37_g_1c8ad7f0_9b2f_4f9a_ba9f_5f6e3b3c2c40_lblDrawDate"}
        )

        if date_element:
            return pd.to_datetime(date_element.text.strip(), errors="coerce")

    except Exception as e:
        print("Website check failed:", e)

    return None


# -----------------------
# CHECK IF DATA UPDATED
# -----------------------

df_check = pd.read_csv(CSV_FILE)

latest_csv_date = pd.to_datetime(df_check["draw_date"]).max()

latest_web_date = get_latest_website_date()

if latest_web_date is not None and latest_csv_date >= latest_web_date:

    print("Data already up to date. Models remain unchanged.")
    exit()

print("New draw detected or website unavailable. Running training pipeline...\n")

# -----------------------
# LOAD DATA
# -----------------------

df = df_check.copy()

number_cols = [c for c in df.columns if c not in ["draw_number","draw_date"]]

numbers=[]

for _,row in df.iterrows():

    draw_nums=[]

    for c in number_cols:
        draw_nums.append(str(row[c]).zfill(4))

    numbers.append(draw_nums)

df["numbers"]=numbers


# -----------------------
# FLATTEN DATA
# -----------------------

all_numbers=[]
draw_idx=[]

for i,row in df.iterrows():

    for n in row["numbers"]:
        all_numbers.append(n)
        draw_idx.append(i)

dataset=pd.DataFrame({
    "draw":draw_idx,
    "number":all_numbers
})

dataset["d1"]=dataset["number"].str[0].astype(int)
dataset["d2"]=dataset["number"].str[1].astype(int)
dataset["d3"]=dataset["number"].str[2].astype(int)
dataset["d4"]=dataset["number"].str[3].astype(int)

dataset["digit_sum"]=dataset[["d1","d2","d3","d4"]].sum(axis=1)
dataset["odd_count"]=(dataset[["d1","d2","d3","d4"]] % 2).sum(axis=1)
dataset["high_count"]=(dataset[["d1","d2","d3","d4"]] >=5).sum(axis=1)

dataset["repeat_count"]=dataset.apply(
    lambda r: len([x for x in [r.d1,r.d2,r.d3,r.d4] if [r.d1,r.d2,r.d3,r.d4].count(x)>1]),
    axis=1
)

# -----------------------
# FEATURE WINDOWS
# -----------------------

windows=[20,50,100,200]

features=[]

for i in range(max(windows),len(dataset)):

    f={}
    row=dataset.iloc[i]

    for w in windows:

        hist=dataset.iloc[i-w:i]

        for digit in range(10):

            for pos in ["d1","d2","d3","d4"]:

                f[f"freq_{digit}_{pos}_w{w}"]=(hist[pos]==digit).mean()

        f[f"sum_mean_w{w}"]=hist["digit_sum"].mean()
        f[f"sum_std_w{w}"]=hist["digit_sum"].std()

        f[f"odd_ratio_w{w}"]=hist["odd_count"].mean()
        f[f"high_ratio_w{w}"]=hist["high_count"].mean()
        f[f"repeat_ratio_w{w}"]=hist["repeat_count"].mean()

    for digit in range(10):

        for pos in ["d1","d2","d3","d4"]:

            prev=dataset.iloc[:i]
            idx=prev[prev[pos]==digit].index

            gap = 999 if len(idx)==0 else i-idx[-1]

            f[f"gap_{digit}_{pos}"]=gap

    prev=dataset.iloc[i-1]

    f["prev_d1"]=prev["d1"]
    f["prev_d2"]=prev["d2"]
    f["prev_d3"]=prev["d3"]
    f["prev_d4"]=prev["d4"]

    f["target_d1"]=row["d1"]
    f["target_d2"]=row["d2"]
    f["target_d3"]=row["d3"]
    f["target_d4"]=row["d4"]

    features.append(f)

features=pd.DataFrame(features).fillna(0)


# -----------------------
# TRAIN XGBOOST
# -----------------------

X=features.drop(columns=["target_d1","target_d2","target_d3","target_d4"])

joblib.dump(X.columns.tolist(), f"{MODEL_DIR}/feature_columns.pkl")

xgb_models={}

for pos in ["d1","d2","d3","d4"]:

    y=features[f"target_{pos}"]

    X_train,X_test,y_train,y_test=train_test_split(
        X,y,test_size=0.2,shuffle=False,random_state=42
    )

    model=XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        tree_method="hist"
    )

    model.fit(X_train,y_train)

    acc=model.score(X_test,y_test)

    print(pos,"XGB accuracy:",acc)

    joblib.dump(model, f"{MODEL_DIR}/xgb_{pos}.pkl")

    xgb_models[pos]=model


# -----------------------
# LSTM MODEL
# -----------------------

seq_len=30

seq_data=dataset[["d1","d2","d3","d4"]].values

X_seq=[]
y_seq=[]

for i in range(seq_len,len(seq_data)):

    X_seq.append(seq_data[i-seq_len:i])
    y_seq.append(seq_data[i])

X_seq=np.array(X_seq)
y_seq=np.array(y_seq)

y_seq_cat=[to_categorical(y_seq[:,i],10) for i in range(4)]

model=Sequential()

model.add(LSTM(64,input_shape=(seq_len,4)))
model.add(Dense(64,activation="relu"))
model.add(Dense(40,activation="softmax"))

model.compile(optimizer="adam",loss="categorical_crossentropy")

y_concat=np.concatenate(y_seq_cat,axis=1)

model.fit(X_seq,y_concat,epochs=10,batch_size=64,verbose=1)

model.save(f"{MODEL_DIR}/lstm_model.keras")


# -----------------------
# PREDICTION FEATURES
# -----------------------

i=len(dataset)

predict={}

for w in windows:

    hist=dataset.iloc[i-w:i]

    for digit in range(10):

        for pos in ["d1","d2","d3","d4"]:

            predict[f"freq_{digit}_{pos}_w{w}"]=(hist[pos]==digit).mean()

    predict[f"sum_mean_w{w}"]=hist["digit_sum"].mean()
    predict[f"sum_std_w{w}"]=hist["digit_sum"].std()

    predict[f"odd_ratio_w{w}"]=hist["odd_count"].mean()
    predict[f"high_ratio_w{w}"]=hist["high_count"].mean()
    predict[f"repeat_ratio_w{w}"]=hist["repeat_count"].mean()

for digit in range(10):

    for pos in ["d1","d2","d3","d4"]:

        idx=dataset[dataset[pos]==digit].index
        gap = 999 if len(idx)==0 else i-idx[-1]

        predict[f"gap_{digit}_{pos}"]=gap

prev=dataset.iloc[-1]

predict["prev_d1"]=prev["d1"]
predict["prev_d2"]=prev["d2"]
predict["prev_d3"]=prev["d3"]
predict["prev_d4"]=prev["d4"]

X_pred=pd.DataFrame([predict]).fillna(0)

# -----------------------
# PROBABILITIES
# -----------------------

digit_probs_xgb={}

for pos in ["d1","d2","d3","d4"]:

    probs=xgb_models[pos].predict_proba(X_pred)[0]
    digit_probs_xgb[pos]=normalize_probs(probs)


seq_input=seq_data[-seq_len:].reshape(1,seq_len,4)

lstm_out=model.predict(seq_input)[0]

lstm_probs={
"d1":normalize_probs(lstm_out[0:10]),
"d2":normalize_probs(lstm_out[10:20]),
"d3":normalize_probs(lstm_out[20:30]),
"d4":normalize_probs(lstm_out[30:40])
}


digit_probs={}

for pos in ["d1","d2","d3","d4"]:

    combined=(digit_probs_xgb[pos]*0.7 + lstm_probs[pos]*0.3)

    digit_probs[pos]=normalize_probs(combined)


# -----------------------
# MONTE CARLO
# -----------------------

predictions=[]

for _ in range(50000):

    d1=np.random.choice(np.arange(10),p=digit_probs["d1"])
    d2=np.random.choice(np.arange(10),p=digit_probs["d2"])
    d3=np.random.choice(np.arange(10),p=digit_probs["d3"])
    d4=np.random.choice(np.arange(10),p=digit_probs["d4"])

    predictions.append(f"{d1}{d2}{d3}{d4}")

ranked=pd.Series(predictions).value_counts()

result=ranked.head(50).reset_index()
result.columns=["number","freq"]

print("\nTop 50 predicted numbers\n")
print(result)

result.to_csv(f"{MODEL_DIR}/top_predictions.csv",index=False)

print("\nModels and predictions saved to /models folder.")

KeyboardInterrupt: 